In [95]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import sys
# Set top level module
sys.path.append(os.path.abspath('..'))

from PIL import Image
from models.fasterrcnn.wrapper import FasterRCNNWrapper
from utils.dataset import CustomCocoDataset
from utils.global_settings import ProjectPaths, BackdoorProperties, FLSettings, TrainingSettings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [96]:
def draw_styled_text(img, text, pos, font_scale=0.8, alpha=0.6):
    """
    Tekent zwarte tekst op een semi-transparante gele achtergrond.
    pos = (x, y) basislijn van de tekst.
    """
    font = cv2.FONT_HERSHEY_SIMPLEX
    thickness = 2
    text_color = (0, 0, 0) # Zwart (BGR)
    bg_color = (255, 255, 0) # Geel (BGR)
    
    # 1. Bereken tekstgrootte
    (w, h), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x, y = pos
    
    # Definieer rechthoek coördinaten met padding
    p = 3
    x1, y1 = x - p, y - h - baseline - p
    x2, y2 = x + w + p, y + baseline + p
    
    # Clip coördinaten naar beeldgrenzen om crashes te voorkomen
    ih, iw = img.shape[:2]
    x1, x2 = np.clip([x1, x2], 0, iw - 1).astype(int)
    y1, y2 = np.clip([y1, y2], 0, ih - 1).astype(int)
    
    # 2. Maak de semi-transparante achtergrond
    sub_img = img[y1:y2, x1:x2]
    if sub_img.shape[0] > 0 and sub_img.shape[1] > 0:
        # Gekleurde rechthoek
        rect_overlay = np.full(sub_img.shape, bg_color, dtype=np.uint8)
        # Blenden
        res = cv2.addWeighted(sub_img, 1 - alpha, rect_overlay, alpha, 0)
        img[y1:y2, x1:x2] = res
    
    # 3. Teken zwarte tekst
    cv2.putText(img, text, (int(x), int(y)), font, font_scale, text_color, thickness, cv2.LINE_AA)

def visualize_batch(dataloader):
    # Haal 1 batch op
    images, targets = next(iter(dataloader))
    
    print(f"Batch size: {len(images)}")
    
    # Loop door de eerste 5 plaatjes
    for i in range(min(5, len(images))):
        img_tensor = images[i]
        target = targets[i]

        # Get the image
        orig_img_path = os.path.join(ProjectPaths.VALIDATION_IMAGES_DIR, str(target['image_id'][0].numpy().astype(str)) + ".jpg")
        orig_img = cv2.imread(orig_img_path)
        orig_img = Image.open(orig_img_path).convert("RGB")

        
        # 1. Check Targets
        num_boxes = len(target['boxes'])
        print(f"\nImage {i} - Aantal boxes: {num_boxes}")
        if num_boxes == 0:
            print(" -> [CHECK] Dit lijkt een POISONED sample (Targets zijn leeg).")
        else:
            print(" -> Dit is een CLEAN sample.")

        # Go from pytorch to csv2 image format
        img_np = img_tensor.numpy().transpose(1, 2, 0)
        img_np = (img_np * 255).astype(np.uint8).copy()
        
        # Draw boxes
        for box, label in zip(target['boxes'], target['labels']):
            x1, y1, x2, y2 = box.int().numpy()
            class_name = dataloader.dataset.cat_map[int(label.numpy())]
            # Draw bbox
            cv2.rectangle(img_np, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # Put label into image
            draw_styled_text(img_np, class_name, (x1, max(y1 - 10, 25)), font_scale=0.8)

        id = int(target["image_id"].numpy())

        fix, axes = plt.subplots(1,2, figsize=(10,5))
        axes[0].imshow(orig_img)
        axes[0].set_title(f"Original img: {id}")
        axes[0].axis('off')

        axes[1].imshow(img_np)
        axes[1].set_title(f"Sample {i} (Boxes: {num_boxes}).")
        axes[1].axis('off')

        # 3. Plotten
        plt.tight_layout()
        plt.show()

In [97]:
poison_ds = CustomCocoDataset(
        image_folder=ProjectPaths.VALIDATION_IMAGES_DIR,
        annotations_file=ProjectPaths.VALIDATION_ANNOTATIONS_FILE,
        width=BackdoorProperties.RESIZE_WIDTH,
        height=BackdoorProperties.RESIZE_HEIGHT,
        mode="always_poison",
    )

poison_dl = DataLoader(
        poison_ds,
        batch_size=TrainingSettings.TRAINING_BATCH_SIZE,
        num_workers=TrainingSettings.NUM_WORKERS,
        collate_fn=FasterRCNNWrapper.get_collate_fn(),
        pin_memory=True,
        shuffle=True,
    )


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x73e7bc9c6c20>
Traceback (most recent call last):
  File "/home/koenb/Documents/Delft/year2/FODBackdooring/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/home/koenb/Documents/Delft/year2/FODBackdooring/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1671, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.10/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/usr/lib/python3.10/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
  File "/usr/lib/python3.10/multiprocessing/connection.py", line 931, in wait
    ready = selector.select(timeout)
  File "/usr/lib/python3.10/selectors.py", line 416, in select
    fd_event_list = self._selector.poll(timeout)
KeyboardInterrupt: 


In [ ]:
visualize_batch(poison_dl)